# Manual vs PSO in Colab

This notebook keeps your local project files unchanged and gives you a Colab-friendly way to run the experiments on a cloud GPU.

Before running:
- In Colab, go to `Runtime -> Change runtime type -> T4 GPU` (or any available GPU).
- Update `REPO_URL` below to your GitHub repository URL.
- If your repo is private, either make it accessible to Colab or upload the project as a zip to Drive and adapt the clone step.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/your-username/COSC4P96-GroupProject.git"
REPO_DIR = Path("/content/COSC4P96-GroupProject")
SAVE_TO_DRIVE = False
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/COSC4P96_outputs"


## Optional: mount Google Drive

Turn `SAVE_TO_DRIVE` on in the cell above if you want your outputs to persist after the session ends.

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print(f"Drive output directory ready: {DRIVE_OUTPUT_DIR}")
else:
    print("Drive mount skipped.")


## Clone the project

In [ ]:
if REPO_DIR.exists():
    print(f"Repo already exists at {REPO_DIR}")
else:
    !git clone "$REPO_URL" "$REPO_DIR"

%cd /content/COSC4P96-GroupProject
!ls

## Install dependencies

Colab already includes a GPU-ready PyTorch stack in most runtimes, so avoid upgrading `torch` or `torchvision` here. This cell only installs plotting support if needed.

In [ ]:
!pip install -q matplotlib

## Verify the GPU

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Make sure Colab runtime is set to GPU.")


## Choose where outputs should go

In [ ]:
OUTPUT_ROOT = DRIVE_OUTPUT_DIR if SAVE_TO_DRIVE else "study_outputs_colab"
print("Output root:", OUTPUT_ROOT)

## Smoke test

Use this first to confirm the full pipeline works in Colab.

In [ ]:
!python compare_manual_vs_pso.py --labeled-ratios 0.1 --trials 1 --max-manual-candidates 1 --pso-swarm-size 1 --pso-iters 1 --tuning-ssl-rounds 1 --tuning-epochs-per-round 1 --final-ssl-rounds 1 --final-epochs-per-round 1 --num-workers 0 --output-root "$OUTPUT_ROOT"

## Medium run

A good balance if you want useful results without committing to the full study.

In [ ]:
!python compare_manual_vs_pso.py --trials 3 --max-manual-candidates 3 --pso-swarm-size 2 --pso-iters 2 --tuning-ssl-rounds 1 --tuning-epochs-per-round 2 --final-ssl-rounds 2 --final-epochs-per-round 3 --num-workers 0 --output-root "$OUTPUT_ROOT"

## Full run

This is the report-quality version, but it may take a long time even on Colab.

In [ ]:
!python compare_manual_vs_pso.py --output-root "$OUTPUT_ROOT"

## Inspect the newest output folder

In [ ]:
root = Path(OUTPUT_ROOT)
folders = sorted([p for p in root.glob("manual_vs_pso_*") if p.is_dir()])
latest = folders[-1] if folders else None
print("Latest folder:", latest)
if latest is not None:
    for item in sorted(latest.iterdir()):
        print(item.name)

## Render a downloadable matplotlib report image

In [ ]:
if latest is not None:
    !python render_study_report.py --study-dir "$latest"
    report_path = latest / "study_report.png"
    print(report_path)
else:
    print("No output folder found yet.")

## Preview and download the report image

In [ ]:
if latest is not None:
    from IPython.display import Image, display
    from google.colab import files

    report_path = latest / "study_report.png"
    display(Image(filename=str(report_path)))
    files.download(str(report_path))
else:
    print("No output folder found yet.")